# Fig. 6 — shape-index heatmaps (panels A–D)

In [ ]:
from vertex_lite.custom import load_data_v3
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import os
import numpy as np
import pyprojroot.here as here

## Load Sort and Filter

In [ ]:
def filter_data(time,local=False,cluster=False):
    if cluster:
        rpath=here('results/resultados_cluster')
    else:
        rpath=here('results/resultados_random')
    if local:
        df=load_data_v3(f'{rpath}/noise_*/pct_*/sim_*/mutant.parquet',columns=['time','shape_neigh'],time_filter=time)
        df=df.groupby(['mesh_type','pct_mutant'])['shape_neigh'].mean().reset_index()
    else:
        df=load_data_v3(f'{rpath}/noise_*/pct_*/sim_*/global.parquet',columns=['time','shape_global'],time_filter=time)
        df=df.groupby(['mesh_type','pct_mutant'])['shape_global'].mean().reset_index()
    if df.empty:
        return print(f"No data found for time {time} and local {local}")

    return df

## Plotting

In [ ]:
def get_heatmap_data(time, local=False, case="shape", cluster=False):
    df = filter_data(time, local, cluster)
    shape_col = 'shape_neigh' if local else 'shape_global'
    title = "Shape Index"
    # mesh_type as columns, pct_mutant as index, shape index as value
    heatmap_data = df.set_index(['mesh_type', 'pct_mutant'])[shape_col].unstack()

    heatmap_data = heatmap_data.sort_index().sort_index(axis=1)
    heatmap_data = heatmap_data.T
    return heatmap_data, title


def plot_heatmap(heatmap_data, title, time, local=False, case=None, cluster=False):
    cmap = sns.diverging_palette(220, 20, as_cmap=True)
    # Colour scale centred on p* = 3.81 (shared by all heatmaps and the Fig. 5 colour bar)
    norm = mcolors.TwoSlopeNorm(vmin=3.72, vcenter=3.81, vmax=3.9)

    sns.set_theme(style="white", context="talk")

    plt.rcParams.update({
        "font.size": 20,
        "axes.labelsize": 25,
        "axes.titlesize": 25,
        "legend.fontsize": 12,
        "axes.linewidth": 0.0,
        "xtick.labelsize": 24,
        "ytick.labelsize": 24,
    })
    fig, ax = plt.subplots(figsize=(7, 5))

    im = ax.imshow(
        heatmap_data.values,
        cmap=cmap,
        origin='lower',  # first row at the bottom
        aspect='auto',
        norm=norm
    )

    # --- AXES LABELS ---
    step = 2
    plt.tick_params(axis='both', labelsize=15)
    # X-axis: mesh distortion (columns)
    x_ticks = np.arange(0, len(heatmap_data.columns), step)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(heatmap_data.columns[::step], rotation=45)

    # Y-axis: % mutants (index)
    y_ticks = np.arange(0, len(heatmap_data.index), step)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(heatmap_data.index[::step])

    # --- COLORBAR ---
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(case)

    ax.annotate(
        f'T={time} | local={local}',
        xy=(0.0, -0.2),
        xycoords='axes fraction',
        ha='left',
        fontsize=10
    )

    plt.xlabel("Initial Mesh Distortion")
    plt.ylabel("% mutants")
    plt.title(f'{title} for Neighbors of Mutant Cells' if local else f'{title} for Whole Tissue')

    plt.tight_layout()

    suffix = "local" if local else "global"
    clustering = "cluster" if cluster else "random"

    savedir = here("results/si/plots/heatmaps")
    os.makedirs(savedir, exist_ok=True)

    filename = f"{savedir}/heatmap_{clustering}_time_{time}_{case}_{suffix}.svg"

    fig.savefig(filename, format='svg')
    plt.close(fig)

## Figures

In [ ]:
# Heatmaps evaluated at t = 15000 (last saved time step, 14980)
times = [[14980], [14980]]

data_cache = {}

for c in ["shape"]:
    for l in [True, False]:
        # Random
        for t in times[0]:
            key = (t, l, c, False)  # (time, local, case, cluster)
            print(f"Analyzing {key}...")
            data_cache[key] = get_heatmap_data(time=t, local=l, case=c, cluster=False)

        # Cluster
        for t in times[1]:
            key = (t, l, c, True)
            print(f"Analyzing {key}...")
            data_cache[key] = get_heatmap_data(time=t, local=l, case=c, cluster=True)

print("All data loaded into memory.")

In [ ]:
for key, (heatmap_df, title) in data_cache.items():
    t, l, c, is_cluster = key
    plot_heatmap(heatmap_df, title, t, local=l, case=c, cluster=is_cluster)